# Claim Adjudication — End-to-End

Invoice → Extraction → Policy retrieval → Agent 1 → Calculation → Agent 2 → Final report

- **SFT + QLoRA:** Fine-tunes the vision model to extract invoice data.
- **DPO + QLoRA:** Fine-tunes the reasoning model for policy selection and review.
- **RAG:** Retrieves policy evidence for each invoice item.
- **Deterministic calculation:** Python applies limits and co-pay; unresolved items remain pending.

## 1. Setup

In [ ]:
from pathlib import Path
import html
import io
import json
import os
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'src/components/agentic_workflow/graph.py').is_file()
            or (p / 'optum_claim_adjudication/src/components/agentic_workflow/graph.py').is_file())
if not (ROOT / 'src').is_dir():
    ROOT = ROOT / 'optum_claim_adjudication'
os.chdir(ROOT)
sys.path[:0] = [str(ROOT), str(ROOT / 'notebooks')]

import pandas as pd
from demo_runner import show, show_html, show_image
from demo_support import check_services, run_workflow, financial_summary

show(pd.DataFrame(check_services(ROOT)))

## 2. Input invoice

In [ ]:
from UI.invoice_date import extract_invoice_date

IMAGE_PATH = ROOT / 'notebooks/demo_artifact/invoice_2024.jpg'
show_image(IMAGE_PATH)
date_metadata = extract_invoice_date(IMAGE_PATH, endpoint='http://127.0.0.1:8000/v1')
show(pd.DataFrame([{
    'Invoice': IMAGE_PATH.name,
    'Invoice date': date_metadata['invoice_date'],
    'Date status': date_metadata['status'],
    'Verification': date_metadata['verification_status'],
}]))
if date_metadata['status'] != 'FOUND':
    raise RuntimeError('Invoice date needs review before the workflow can run.')

## 3. Run workflow · Extract and validate

In [ ]:
demo = run_workflow(ROOT, IMAGE_PATH, date_metadata)
final_report = demo.report
request_id = demo.request_id

validated = demo.stages['csv_validation']
if not validated.get('csv_validated'):
    raise RuntimeError('Invoice CSV validation failed.')
invoice = validated['invoice']
show(pd.read_csv(io.StringIO(validated['extracted_csv']), dtype=str, keep_default_na=False))
show(pd.DataFrame([{
    'Extraction model': final_report['metadata']['extraction']['returned_model'],
    'CSV validation': 'Passed',
    'Invoice items': len(invoice.lines),
}]))

## 4. Retrieved policy evidence

In [ ]:
rag = demo.stages['policy_retrieval']
evidence = [
    {'Item': invoice.lines[int(index)].item,
     'Policy': rule['rule_id'],
     'Category': rule['category'],
     'Conditions / evidence': rule['conditions']}
    for index, rules in rag['candidate_rules'].items()
    for rule in rules
]
show(pd.DataFrame(evidence))
show(pd.DataFrame([{'Retrieval status': rag['rag_status'], 'Evidence source': 'Milvus policy index'}]))

## 5. Agent 1 · Policy selection

In [ ]:
a1 = demo.stages['policy_adjudication_agent']
selections = {row['item_index']: row['suggested_status']
              for row in a1['metadata']['agent_1']['model_selections']}
show(pd.DataFrame([
    {'Item': invoice.lines[row.item_index].item,
     'Policy': ', '.join(row.policy_ids),
     'Model suggestion': selections.get(row.item_index, '—'),
     'Validated decision': row.suggested_status,
     'Reason': row.reason}
    for row in a1['agent_suggestions'].items
]))

## 6. Calculation · Amounts in INR

In [ ]:
calculation = demo.stages['deterministic_adjudication']['claim_result'].model_dump(mode='json')
show(pd.DataFrame(calculation['decisions'])[[
    'item', 'billed_amount', 'approved_amount', 'disallowed_amount',
    'pending_review_amount', 'status',
]].rename(columns={
    'item': 'Item', 'billed_amount': 'Billed', 'approved_amount': 'Eligible',
    'disallowed_amount': 'Disallowed', 'pending_review_amount': 'Pending review',
    'status': 'Decision',
}))

## 7. Agent 2 · Evidence review

In [ ]:
a2 = demo.stages['reasoning_review_agent']
if 'agent_review' not in a2:
    raise RuntimeError('Agent 2 did not return an accepted evidence review.')
show(pd.DataFrame([
    {'Item': invoice.lines[row.item_index].item,
     'Decision': row.status,
     'Review needed': 'Yes' if row.needs_review else 'No',
     'Evidence / explanation': row.explanation}
    for row in a2['agent_review'].items
]))
assert a2['claim_result'].model_dump(mode='json') == calculation, 'Review changed calculated amounts.'

## 8. Final result and report

In [ ]:
from UI.claim_report import generate_claim_pdf

claim = final_report['claim']
show(pd.DataFrame([{
    'Workflow status': final_report['status'],
    'Policy': claim['policy_id'],
    'Agent reviewed': final_report['reviewed'],
}]))
show(pd.DataFrame(financial_summary(claim)))
if claim['claim_status'] == 'manual_review':
    assert claim['final_reimbursement'] is None
    show_html('<p style="background:#fff3cd;color:#664d03;padding:14px">'
              '<b>Review required — final reimbursement is not finalized.</b></p>')

pdf_path = demo.output_dir / 'report.pdf'
generate_claim_pdf(final_report, date_metadata, pdf_path)
assert pdf_path.read_bytes().startswith(b'%PDF-')
links = [f'<a style="color:#0057a8" href="{html.escape((demo.output_dir / name).relative_to(ROOT / "notebooks").as_posix())}" target="_blank">{label}</a>'
         for name, label in [('report.pdf', 'PDF report'), ('final_report.json', 'Full JSON'),
                             ('validated_invoice.csv', 'Validated CSV')]]
show_html(' &nbsp; | &nbsp; '.join(links), str(demo.output_dir))

## 9. Execution summary

In [ ]:
show(pd.DataFrame([
    {'Stage': row['node'].replace('_', ' ').title(),
     'Outcome': row['outcome'], 'Time (seconds)': round(row['elapsed_ms'] / 1000, 3)}
    for row in final_report['trace']
]))
execution_summary = {
    'Request ID': request_id,
    'Executed at UTC': demo.executed_at,
    'Workflow time (seconds)': round(demo.duration, 2),
    'Workflow status': final_report['status'],
    'Errors': len(final_report['errors']),
}
show(pd.DataFrame([execution_summary]))
if final_report['warnings']:
    show(pd.DataFrame({'Workflow note': final_report['warnings']}))
(demo.output_dir / 'execution_summary.json').write_text(json.dumps(execution_summary, indent=2) + '\n')